# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### RunnableLambda
일반 Python 함수를 lcel 체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [ ]:
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕 만나서 반갑다~')

In [ ]:
# batch() : 여러 건의 입력을 일괄처리해줌 
runnable.batch(['안녕 만나서 반갑다~', '너도? 나도!', '?!', '😂😂😂😂'])

In [ ]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 +32

celsius_temps = [0, 25 , 100, -10, 37, 36.5]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temps)

In [ ]:
import time     # 출력 딜레이용

def generator(x):
    for y in x:     # 입력을 문자 단위로 순회
        yield y     # 한 글자씩 반환(스트리밍 방식)
        
runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세용~🤣😊😅😂안녕하세용~🤣😊😅😂안녕하세용~🤣😊😅😂안녕하세용~🤣😊😅😂안녕하세용~🤣😊😅😂'):
    print(chunk, end='', flush=True)    # chunk를 줄바꿈 없이 즉시 출력
    time.sleep(0.1)     # 글자 출력마다 딜레이 0.1초

In [ ]:
def gen(x):
    for y in x:
        yield y
        
gen10 = gen(range(10))

for n in gen10:
    print(n)

In [ ]:
next(gen10) # 재너레이터 다음값 1개 반환(다 꺼내고 나면 StopIteration 발생)

### RunnableSequence
Runnable 객체를 순차연결 해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x]*3 )

chain = RunnableSequence(runnable1, runnable2)
chain.invoke(3)

In [ ]:
chain = runnable1 | runnable2
chain.invoke(3)

In [ ]:
chain = runnable2 | runnable1
chain.invoke(3)

### RunnableParallel
여러 Runnable 객체를 인자로 받아, 병렬처리후 각각의 응답을 하나의 dict로 반환

In [ ]:
from langchain_core.runnables import RunnableParallel   # 여러 Runnable들을 같은 입력으로 병렬로 실행

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x]*3 )

chain = RunnableParallel(r1 = runnable1, r2 = runnable2)    # r1, r2를 병렬 실행해 dict 로 반환
chain.invoke(3)

- 사용자가 준 주제를 이용해 삼행시, 농답, 시를 각각 생성해서 하나의 응답으로 반환

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제: {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser

joke_prompt = PromptTemplate.from_template(
    '당신은 한국식 농담계의 엄청난 고수입니다. 다음 주제로 배꼽이 빠질만한 농담을 지어주세요. 주제: {topic}'
)
joke_chain = joke_prompt | llm | output_parser

poem_prompt = PromptTemplate.from_template(
    '당신은 현대시의 엄청난 작가입니다. 다음 주제로 눈물이 나올 정도의 감성적인 시를 지어주세요. 주제: {topic}'
)
poem_chain = poem_prompt | llm | output_parser

chain = RunnableParallel(
    acrostic_poem = n_poem_chain,
    joke = joke_chain,
    poem = poem_chain
)

def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']
    return f"""
    n행시 :
    {acrostic_poem}

    농담 :
    {joke}

    현대시 : 
    {poem}
    """
    
chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic': '북아메리카'}))

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template(
    '당신은 n행시의 고수입니다. 다음주제로 맛깔나는 n행시로 지어주세요. 주제: {topic}'
)
chain = {'topic': RunnablePassthrough()} | prompt | llm | output_parser

print(chain.invoke('부트캠프'))

In [ ]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template("""
당신은 n행시의 고수입니다. 다음주제로 맛깔나는 n행시로 지어주세요. 주제: {topic}

주제 : {topic}

출력형식 : 
=== <주제> <n행시>===
<n행시 작성>    
"""
)
chain = ({'topic': RunnablePassthrough()}
        | RunnablePassthrough.assign(
            n = lambda x : len(x['topic']),
            k = lambda x: 100
        )
        | prompt
        | llm
        | output_parser)
        
print(chain.invoke('코카콜라'))